In [1]:
%store -r

In [2]:
import json
import os.path

import commute_dm.core
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

In [4]:
def write_interface_to_json_file(interface, output_file_path, description):
    """Serialize a `commute_dm.core.get_interface` result to a JSON file.

    `interface` is `{uniprot_id: [{"collection": <Collection node>,
    "entry": <CollectionEntry node>, "node": <Protein node>}]}`. The join itself
    lives in `commute_dm.core.get_interface`; this notebook only archives it.
    """
    data = []
    for identifier, elements in interface.items():
        data.append(
            {
                "annotation": {"namespace": "uniprot", "identifier": identifier},
                "model_elements": [
                    {
                        "collection": element["collection"]["name"],
                        "entry_file_path": element["entry"]["file_path"],
                        "protein_element_id": element["node"].element_id,
                        "protein_name": element["node"]["name"],
                    }
                    for element in elements
                ],
            }
        )
    output = {"description": description, "data": data}
    with open(output_file_path, "w") as f:
        json.dump(output, f)

In [5]:
COVID_PD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_pd.json")
COVID_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_ad.json")
COVID_PD_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_pd_ad.json")
COVID_AF_PD_AF_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_af_pd_af_ad.json")

We remake the directory where we store the interfaces:

In [6]:
commute_dm.utils.remake_dir(INTERFACE_DIR)

## Computing the interface between the maps based on annotations

### Interface between the COVID DM CD and the PD DM CD

We compute the interface between the COVID and PD maps based on shared UniProt annotations, i.e., we query every UniProt id such that:
- the id is carried by a `:Protein` annotation key (`urn:miriam:uniprot:<id>`) in the `COVID_DM_CD` collection, **and**
- the same id is carried by a `:Protein` in the `PD_DM_CD` collection.

For each such id we collect all the proteins (with their collection and entry) that carry it.

In [7]:
interface = commute_dm.core.get_interface(session, ["COVID_DM_CD", "PD_DM_CD"])

We save the interface to a JSON file:

In [8]:
write_interface_to_json_file(
    interface,
    COVID_PD_INTERFACE_FILE,
    (
        "Interface between the COVID maps and the PD maps, based on shared UniProt annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins carrying a UniProt annotation that is shared\n"
        "(same UniProt id) between the COVID_DM_CD and PD_DM_CD collections."
    ),
)

### Interface between the COVID DM CD and the AD KG BEL

We compute the interface between the COVID maps and the AD KG based on shared UniProt annotations, i.e., we query every UniProt id carried by a `:Protein` annotation key in **both** the `COVID_DM_CD` and `AD_KG_BEL` collections. The AD KG proteins are now matched through their UniProt annotations (added in `2_00`), so we no longer read inside the BEL nodes.

In [9]:
interface = commute_dm.core.get_interface(session, ["COVID_DM_CD", "AD_KG_BEL"])

We save the interface to a JSON file:

In [10]:
write_interface_to_json_file(
    interface,
    COVID_AD_INTERFACE_FILE,
    (
        "Interface between the COVID maps and the AD KG (from Fraunhofer), based on shared UniProt annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins carrying a UniProt annotation that is shared\n"
        "(same UniProt id) between the COVID_DM_CD and AD_KG_BEL collections."
    ),
)

### Interface between the COVID DM CD, the PD DM CD, and the AD KG BEL

We compute the interface between the COVID maps, the PD maps, and the AD KG based on shared UniProt annotations, i.e., we query every UniProt id carried by a `:Protein` annotation key in **all three** of the `COVID_DM_CD`, `PD_DM_CD` and `AD_KG_BEL` collections.

In [11]:
interface = commute_dm.core.get_interface(
    session, ["COVID_DM_CD", "PD_DM_CD", "AD_KG_BEL"]
)

In [12]:
len(interface)

127

We save the interface to a JSON file:

In [13]:
write_interface_to_json_file(
    interface,
    COVID_PD_AD_INTERFACE_FILE,
    (
        "Interface between the COVID maps, the PD maps, and the AD KG (from Fraunhofer),\n"
        "based on shared UniProt annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins carrying a UniProt annotation that is shared\n"
        "(same UniProt id) across the COVID_DM_CD, PD_DM_CD and AD_KG_BEL collections."
    ),
)

### Interface between the COVID DM CD AF, the PD DM CD AF, and the AD KG BEL

We compute the interface between the COVID AF maps, the PD AF maps, and the AD KG based on shared UniProt annotations, i.e., we query every UniProt id carried by a `:Protein` annotation key in **all three** of the `COVID_DM_CD_AF`, `PD_DM_CD_AF` and `AD_KG_BEL` collections.

In [14]:
interface = commute_dm.core.get_interface(
    session, ["COVID_DM_CD_AF", "PD_DM_CD_AF", "AD_KG_BEL"]
)

In [15]:
len(interface)

127

We save the interface to a JSON file:

In [16]:
write_interface_to_json_file(
    interface,
    COVID_AF_PD_AF_AD_INTERFACE_FILE,
    (
        "Interface between the COVID AF maps, the PD AF maps, and the AD KG (from Fraunhofer),\n"
        "based on shared UniProt annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins carrying a UniProt annotation that is shared\n"
        "(same UniProt id) across the COVID_DM_CD_AF, PD_DM_CD_AF and AD_KG_BEL collections."
    ),
)